# Task 1, Step 1: Unchanged FP16 Baseline and Measurement Protocol

**Model:** `Qwen/Qwen2.5-1.5B` (base, FP16, SDPA attention), revision locked by `setup_data.py`.

This notebook
1. prepares the locked public data (`setup_data.py --public`, first run only, C4 streaming is slow),
2. records the environment and checkpoint revision,
3. tests that the 7 x 28 eligible projections and the tied embedding are enumerated correctly,
4. saves every selected ID (calibration, development, evaluation windows, HellaSwag order),
5. evaluates the FP16 baseline (WikiText-2 + C4 perplexity, 2,000 HellaSwag, storage, latency, memory).

Every later condition reuses exactly this evaluation code (`src/evaluate.py`, `src/runner.py`).

In [ ]:
# --- Setup: locate (or clone) the PA4 repo, pin library versions, prepare data -------------
import os, sys, subprocess, importlib.metadata as md
REPO_URL = "https://github.com/moizejaz-31/ATML-PA4.git"   # change if your repo lives elsewhere

def _find_root(d):
    d = os.path.abspath(d)
    while True:
        if os.path.isdir(os.path.join(d, "shared")) and os.path.isdir(os.path.join(d, "task1_compression")):
            return d
        if os.path.dirname(d) == d:
            return None
        d = os.path.dirname(d)

ROOT = _find_root(os.getcwd())
if ROOT is None:                                   # fresh Kaggle / Colab runtime
    subprocess.run(["git", "clone", "-q", REPO_URL, "ATML-PA4"], check=True)
    ROOT = os.path.abspath("ATML-PA4")
os.chdir(ROOT)
sys.path.insert(0, ROOT)

def _ver(p):
    try:
        return md.version(p)
    except md.PackageNotFoundError:
        return None

if _ver("transformers") != "4.57.1" or _ver("huggingface_hub") != "0.36.2":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==4.57.1", "huggingface_hub==0.36.2",
                    "datasets>=3,<5"], check=True)
    print(">>> Installed pinned versions. Restart the kernel, then run this cell again.")

from shared.src.nb import init_notebook
ctx = init_notebook("task1", need_public=True, need_prefix_banks=False)

In [ ]:
import json
import pandas as pd
import torch
from shared.src.io import save_json
from shared.src.env import environment_record
from task1_compression.src.runner import setup_task1
from task1_compression.src.model_utils import inventory, eligible_linears
from task1_compression.src.quant import dense_fp16_bytes, model_packed_bytes, uniform_assignment
from task1_compression.src.prune import sparse16_storage_bytes
from task1_compression.src.data import HELLASWAG_FORMAT

T = setup_task1(ctx)
model, tok = T.model, T.tok
env = environment_record(dict(model="Qwen/Qwen2.5-1.5B", model_revision=model._pa4_revision,
                              tokenizer=type(tok).__name__, dtype="float16",
                              attn_implementation=model.config._attn_implementation,
                              hellaswag_format=HELLASWAG_FORMAT))
save_json(ctx.results / "environment.json", env)
env

## Eligible-projection and tied-weight enumeration test

In [ ]:
inv = inventory(model)
assert inv["n_layers"] == 28 and inv["n_eligible_matrices"] == 196
assert inv["tied_embeddings"], "Qwen2.5-1.5B ties lm_head to the input embedding; it must be counted once"
expected = {"q_proj": (1536, 1536), "k_proj": (256, 1536), "v_proj": (256, 1536), "o_proj": (1536, 1536),
            "gate_proj": (8960, 1536), "up_proj": (8960, 1536), "down_proj": (1536, 8960)}
assert inv["shapes"] == expected, inv["shapes"]
save_json(ctx.results / "inventory.json", inv)
pd.Series({k: v for k, v in inv.items() if k != "shapes"})

## Saved selections (IDs, windows, seeds)

In [ ]:
assert not set(T.calib_rows) & set(T.dev_rows), "calibration and development windows must be disjoint"
splits = dict(seed=6304,
              calibration_c4_train_rows=T.calib_rows, development_c4_train_rows=T.dev_rows,
              wikitext2_windows=[dict(source_row=w["source_row"], token_offset=w["token_offset"], scored=w["scored_tokens"]) for w in T.ev["wikitext2"]],
              c4_validation_windows=[dict(source_row=w["source_row"], token_offset=w["token_offset"], scored=w["scored_tokens"]) for w in T.ev["c4"]],
              hellaswag_orders_full=[d["order"] for d in T.ev["hellaswag"]],
              hellaswag_diagnostic_subset="first 512 orders of the 2,000-ID manifest",
              sensitivity_probe_subset="first 32 of the 128 calibration windows (same IDs for every probe)")
save_json(ctx.splits / "task1_selected_ids.json", splits)
print({k: (len(v) if isinstance(v, list) else v) for k, v in splits.items()})

## FP16 baseline (all suites, storage, latency, memory)

In [ ]:
def fp16_storage(m, stats):
    return dict(packed=model_packed_bytes(m, {n: 16 for n, *_ in eligible_linears(m)}))

base = T.runner.run("fp16", apply_fn=None, storage_fn=fp16_storage, meta=dict(description="unchanged FP16 baseline"))
pd.DataFrame({k: base[k] for k in ("wikitext2", "c4")}).T[["n_scored_tokens", "n_windows", "mean_nll", "ppl"]]

In [ ]:
pd.Series(dict(hellaswag_n=base["hellaswag"]["n"], acc=base["hellaswag"]["acc"], acc_norm=base["hellaswag"]["acc_norm"]))

In [ ]:
res = base["resources"]
pd.DataFrame({k: {kk: vv for kk, vv in v.items() if kk != "all_ms"} for k, v in res.items() if isinstance(v, dict)}).T

## Storage representations (logical bytes, not GPU allocation)
Dense FP16 counts the tied embedding once. The specified unstructured-sparse format stores 1 mask bit per eligible
weight (byte-aligned per matrix) plus 16 bits per nonzero; quantized formats include scales and zero-points.

In [ ]:
n_mats = [m.weight.numel() for *_, m in eligible_linears(model)]
rows = {"dense FP16": dense_fp16_bytes(model)}
for s in (0.25, 0.5, 0.75):
    rows[f"sparse16 @ {int(s*100)}%"] = sparse16_storage_bytes(n_mats, [int(round((1 - s) * n)) for n in n_mats], inv["n_other_params_unique"])["total_bytes"]
for b in (8, 4, 3, 2):
    rows[f"INT{b} g128 packed"] = model_packed_bytes(model, uniform_assignment(model, b))["total_bytes"]
st = pd.Series(rows) / 2**20
st.to_frame("MiB")